# 01 — Bronze Ingest

Read raw multi-format files from `landing/`, add ingestion metadata, write **Delta** tables to `lakehouse/bronze/`.

## Bronze rules

- **Read as-is.** No schema enforcement, no dedupe, no filtering. Preserve the raw values exactly.
- **Add metadata only.** `_ingest_ts` (when this run happened) and `_source_file` (where the row came from).
- **Always overwrite for now.** This is a learning project — incremental loads come in later notebooks.
- **Delta format.** Gives us ACID and the option to time-travel later.

## Setup

In [ ]:
import sys
from pathlib import Path

if (Path.cwd() / "conf").exists():
    PROJECT_ROOT = Path.cwd()
elif (Path.cwd() / "project" / "conf").exists():
    PROJECT_ROOT = Path.cwd() / "project"
else:
    raise RuntimeError("Run from inside project/ or repo root")
sys.path.insert(0, str(PROJECT_ROOT))

from datetime import datetime
from pyspark.sql import functions as F
from conf.spark import get_spark, LANDING_DIR, BRONZE_DIR

spark = get_spark("BronzeIngest")
INGEST_TS = datetime.utcnow()
print(f"INGEST_TS: {INGEST_TS.isoformat()}")
print(f"LANDING:   {LANDING_DIR}")
print(f"BRONZE:    {BRONZE_DIR}")

In [ ]:
def land_to_bronze(reader, source_path: Path, table: str):
    """Read raw `source_path` with `reader`, add bronze metadata, overwrite Delta."""
    df = (reader(str(source_path))
          .withColumn("_ingest_ts", F.lit(INGEST_TS).cast("timestamp"))
          .withColumn("_source_file", F.input_file_name()))
    target = BRONZE_DIR / table
    (df.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true")
       .save(str(target)))
    n = spark.read.format("delta").load(str(target)).count()
    print(f"  bronze.{table:24s}  {n:>5} rows -> {target}")

BRONZE_DIR.mkdir(parents=True, exist_ok=True)

## CSV sources — header inferred, all columns as strings

In [ ]:
csv_reader = lambda p: spark.read.option("header", True).csv(p)

land_to_bronze(csv_reader, LANDING_DIR / "customers.csv",       "customers")
land_to_bronze(csv_reader, LANDING_DIR / "card_accounts.csv",   "card_accounts")
land_to_bronze(csv_reader, LANDING_DIR / "loan_repayments.csv", "loan_repayments")
land_to_bronze(csv_reader, LANDING_DIR / "bank_accounts.csv",   "bank_accounts")

## JSON sources (JSONL)

In [ ]:
json_reader = lambda p: spark.read.json(p)

land_to_bronze(json_reader, LANDING_DIR / "loan_accounts.json", "loan_accounts")
land_to_bronze(json_reader, LANDING_DIR / "payments.json",      "payments")

## Parquet sources (already typed)

In [ ]:
parquet_reader = lambda p: spark.read.parquet(p)

land_to_bronze(parquet_reader, LANDING_DIR / "card_transactions",    "card_transactions")
land_to_bronze(parquet_reader, LANDING_DIR / "account_transactions", "account_transactions")

## Verify — peek at one bronze table and inspect the schema

In [ ]:
df = spark.read.format("delta").load(str(BRONZE_DIR / "loan_accounts"))
print("Bronze schema for loan_accounts (note customer_id arrived as long from JSON):")
df.printSchema()
df.show(5, truncate=False)

In [ ]:
df = spark.read.format("delta").load(str(BRONZE_DIR / "customers"))
print("Bronze schema for customers (CSV — every column is a string):")
df.printSchema()
df.select("customer_id", "full_name", "email", "city", "_ingest_ts").show(5, truncate=False)

## Summary

All 8 source files are now Delta tables under `lakehouse/bronze/`. Each row has `_ingest_ts` and `_source_file` so you can audit provenance later. Continue with **02-silver-clean.ipynb**.